In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read bronze stock data
print("📖 Reading bronze stock data...")
bronze_df = spark.table("financial_market.bronze.stocks")

print(f"Initial record count: {bronze_df.count():,}")

# 1. Convert string dates to proper date/timestamp types
print("\n🔄 Converting date columns to proper types...")
cleaned_df = bronze_df \
    .withColumn("date", F.to_date(F.col("date"))) \
    .withColumn("fetched_at", F.to_timestamp(F.col("fetched_at"))) \
    .withColumn("run_date", F.to_date(F.col("run_date")))

# 2. Remove records with null critical values
print("\n🧹 Removing records with null critical values...")
initial_count = cleaned_df.count()
cleaned_df = cleaned_df.filter(
    F.col("ticker").isNotNull() &
    F.col("date").isNotNull() &
    F.col("open").isNotNull() &
    F.col("high").isNotNull() &
    F.col("low").isNotNull() &
    F.col("close").isNotNull() &
    F.col("volume").isNotNull()
)
null_removed = initial_count - cleaned_df.count()
print(f"  Removed {null_removed:,} records with null values")

# 3. Remove records with invalid price data
print("\n🔍 Removing records with invalid price data...")
initial_count = cleaned_df.count()
cleaned_df = cleaned_df.filter(
    (F.col("open") > 0) &
    (F.col("high") > 0) &
    (F.col("low") > 0) &
    (F.col("close") > 0) &
    (F.col("volume") >= 0) &
    (F.col("high") >= F.col("low")) &
    (F.col("high") >= F.col("open")) &
    (F.col("high") >= F.col("close")) &
    (F.col("low") <= F.col("open")) &
    (F.col("low") <= F.col("close"))
)
invalid_removed = initial_count - cleaned_df.count()
print(f"  Removed {invalid_removed:,} records with invalid prices")

# 4. Remove duplicate records (keep most recent fetch)
print("\n🔄 Removing duplicates (keeping most recent fetch)...")
initial_count = cleaned_df.count()
window_spec = Window.partitionBy("ticker", "date").orderBy(F.col("fetched_at").desc())
cleaned_df = cleaned_df \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")
duplicates_removed = initial_count - cleaned_df.count()
print(f"  Removed {duplicates_removed:,} duplicate records")

# 5. Add calculated columns for analysis
print("\n➕ Adding calculated columns...")
cleaned_df = cleaned_df \
    .withColumn("daily_range", F.col("high") - F.col("low")) \
    .withColumn("daily_change", F.col("close") - F.col("open")) \
    .withColumn("daily_change_pct", 
                F.round((F.col("close") - F.col("open")) / F.col("open") * 100, 2)) \
    .withColumn("price_volatility", 
                F.round((F.col("high") - F.col("low")) / F.col("open") * 100, 2))

# 6. Add data quality flag
print("\n✅ Adding data quality indicators...")
cleaned_df = cleaned_df \
    .withColumn("data_quality", F.lit("clean")) \
    .withColumn("processed_at", F.current_timestamp())

# Display summary statistics
print("\n📊 Summary Statistics:")
print(f"Final record count: {cleaned_df.count():,}")
print(f"Date range: {cleaned_df.agg(F.min('date')).collect()[0][0]} to {cleaned_df.agg(F.max('date')).collect()[0][0]}")
print(f"Unique tickers: {cleaned_df.select('ticker').distinct().count()}")

# Show sample
print("\n📋 Sample of cleaned data:")
display(cleaned_df.orderBy(F.col("date").desc()).limit(10))

In [0]:
# Silver table mein save karo
print("💾 Saving to Silver layer...")

cleaned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker") \
    .saveAsTable("financial_market.silver.stocks")

print(f"✅ Saved to: financial_market.silver.stocks")
print(f"✅ Partitioned by: ticker")

# verify
df_verify = spark.sql("""
    SELECT
        ticker,
        COUNT(*)                    as total_rows,
        MIN(date)                   as earliest_date,
        MAX(date)                   as latest_date,
        ROUND(AVG(close), 2)        as avg_close,
        ROUND(AVG(daily_change_pct), 2) as avg_daily_change_pct
    FROM financial_market.silver.stocks
    GROUP BY ticker
    ORDER BY ticker
""")

print(f"\n✅ Silver table verification:")
print(f"✅ Total tickers: {df_verify.count()}")
display(df_verify)